# Chapter 22: Intro to PyTorch

Companion Colab notebook for **AI/ML: Zero to Hero**, Module 5, Chapter 22.

This notebook actually *runs* the PyTorch code that the lesson page (`dl/ch22-intro-to-pytorch.html`) shows as reference-only (Pyodide/WebAssembly can't execute PyTorch in the browser). Every cell below was executed for real — no fabricated output.

Chapter 21 built a neural network's forward pass, loss, and backpropagation entirely by hand with NumPy. PyTorch automates every part of that except the architecture design: **tensors** replace NumPy arrays (with GPU support built in), **autograd** computes every gradient for you, and **nn.Module** + **optim** turn the training loop into a few clean lines.


In [1]:
import torch
print('torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())


torch version: 2.8.0+cpu
CUDA available: False


## 22.1 — Tensors: NumPy Arrays That Can Use a GPU

A PyTorch **tensor** is, operation-for-operation, almost identical to a NumPy array — same indexing, same broadcasting, same `@` for matrix multiply. The difference: a tensor can live on a GPU (`.to('cuda')`) for massively parallel computation, and it can track gradients automatically.

In [2]:
import torch
import numpy as np

# NumPy, from Chapter 8
arr = np.array([[1., 2.], [3., 4.]])

# PyTorch — same shape, same operations, same broadcasting rules
t = torch.tensor([[1., 2.], [3., 4.]])
print(t.shape)          # torch.Size([2, 2]) — same idea as arr.shape
print(t @ t)            # matrix multiply, exactly like NumPy's @
print(t.numpy())        # convert back to a NumPy array any time
print(torch.from_numpy(arr))   # and the reverse


torch.Size([2, 2])
tensor([[ 7., 10.],
        [15., 22.]])
[[1. 2.]
 [3. 4.]]
tensor([[1., 2.],
        [3., 4.]], dtype=torch.float64)


## 22.2 — Autograd: Gradients Without Deriving Them

Set `requires_grad=True` on a tensor and PyTorch builds a computation graph as you use it. Call `.backward()` on a final scalar (like a loss) and PyTorch walks that graph backward, filling in `.grad` on every tensor that needed one — this is Chapter 21's entire hand-written backprop section, automated.

In [3]:
import torch

x = torch.tensor(3.0, requires_grad=True)
y = x**2 + 2*x + 1       # y = (x+1)^2, dy/dx = 2x + 2 = 8 when x=3

y.backward()               # computes dy/dx automatically
print(x.grad)               # tensor(8.) — matches the hand-derived answer


tensor(8.)


> **This is the same idea as Chapter 10's numerical derivative** — `(f(x+h) - f(x)) / h` — except autograd computes the EXACT symbolic derivative by tracking every operation, not an approximation. It's the single feature that makes training million-parameter networks practical.

## 22.3 — Building a Model With nn.Module

`nn.Module` is PyTorch's base class for any model. You define the layers in `__init__` and the forward pass in `forward()` — PyTorch wires up autograd for every parameter automatically. This is Chapter 21's `W1, b1, W2, b2`, organized into a reusable class.

In [4]:
import torch
import torch.nn as nn

class TinyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(2, 4)   # 2 inputs -> 4 hidden units (like W1, b1)
        self.layer2 = nn.Linear(4, 1)   # 4 hidden -> 1 output (like W2, b2)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.sigmoid(self.layer1(x))
        x = self.sigmoid(self.layer2(x))
        return x

model = TinyNet()
print(model)
print(f'Total parameters: {sum(p.numel() for p in model.parameters())}')


TinyNet(
  (layer1): Linear(in_features=2, out_features=4, bias=True)
  (layer2): Linear(in_features=4, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)
Total parameters: 17


## 22.4 — Loss Functions & Optimizers

`nn.MSELoss()` / `nn.BCELoss()` replace Chapter 21's hand-written loss formulas. `optim.SGD` or `optim.Adam` replace the manual `W -= lr * d_W` update line — Adam in particular adapts its own learning rate per-parameter and is the default choice for most modern training.

In [5]:
import torch.nn as nn
import torch.optim as optim

model = TinyNet()
criterion = nn.MSELoss()                              # replaces np.mean((y-pred)**2)
optimizer = optim.Adam(model.parameters(), lr=0.01)   # replaces manual W -= lr * d_W
print(criterion)
print(optimizer)


MSELoss()
Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.01
    maximize: False
    weight_decay: 0
)


## 22.5 — The PyTorch Training Loop

Every PyTorch training loop follows the same five lines, every time — memorize this shape, it doesn't change between a tiny 2-layer net and a billion-parameter transformer:

`optimizer.zero_grad() → forward pass → compute loss → loss.backward() → optimizer.step()`

Below we train `TinyNet` on the XOR dataset from Chapter 21 (2000 epochs) and watch the loss actually drop.

In [6]:
import torch

torch.manual_seed(42)
model = TinyNet()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

X = torch.tensor([[0.,0.],[0.,1.],[1.,0.],[1.,1.]])
y = torch.tensor([[0.],[1.],[1.],[0.]])   # XOR — same dataset as Chapter 21

for epoch in range(2000):
    optimizer.zero_grad()        # 1. clear old gradients (they accumulate otherwise!)
    predictions = model(X)       # 2. forward pass
    loss = criterion(predictions, y)   # 3. compute loss
    loss.backward()              # 4. backward pass — autograd fills in every .grad
    optimizer.step()             # 5. update every parameter using its gradient

    if epoch % 500 == 0:
        print(f'Epoch {epoch}, loss {loss.item():.4f}')

print(f'Final loss: {loss.item():.4f}')
print('Rounded predictions:', predictions.round().detach().view(-1).tolist())


Epoch 0, loss 0.2866


Epoch 500, loss 0.0095


Epoch 1000, loss 0.0017


Epoch 1500, loss 0.0007


Final loss: 0.0004
Rounded predictions: [0.0, 1.0, 1.0, 0.0]


> **Forgetting `optimizer.zero_grad()` is the single most common PyTorch bug.** Gradients ACCUMULATE (add up) across calls to `.backward()` by default — skip the reset and your gradients silently grow every epoch, corrupting training in a way that's very confusing to debug.

## 22.6 — GPU Acceleration

Moving a model and its data to a GPU is one line each — `.to(device)`. Everything else in the training loop is unchanged, because PyTorch's tensor operations work identically whether they run on CPU or GPU.

In [7]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = TinyNet().to(device)
X = X.to(device)
y = y.to(device)
# the training loop from 22.5 is IDENTICAL from here — same code, runs on GPU if available
print(f'Training on: {device}')


Training on: cpu


> **In Colab: Runtime → Change runtime type → GPU** gives you a free (if usage-limited) GPU. For this tiny network the speed difference won't be noticeable — but for Chapter 23's CNN it will be.

## Chapter 22 Summary

1. **Tensors** are NumPy arrays that can run on a GPU and track gradients.
2. **Autograd** (`.backward()`) computes exact gradients automatically — no more hand-deriving backprop.
3. **nn.Module** organizes a model's layers and forward pass into a reusable class.
4. **nn.MSELoss/BCELoss + optim.SGD/Adam** replace hand-written loss formulas and weight-update lines.
5. The training loop is always: **zero_grad → forward → loss → backward → step**, five lines, every model.

---

## Mini Project: XOR in PyTorch

Rebuild Chapter 21's hand-written XOR network as real PyTorch code and confirm you get the same result (loss near zero, predictions `[0, 1, 1, 0]`).

**Spec:**
- `TinyNet(nn.Module)` with `Linear(2,4) → Sigmoid → Linear(4,1) → Sigmoid`
- `X` = XOR inputs, `y` = XOR labels, both as torch tensors
- `criterion = nn.MSELoss()`, `optimizer = optim.Adam(model.parameters(), lr=0.05)`
- Train for 2000 epochs using the 5-line loop from 22.5
- Print the final loss and the rounded predictions — confirm they match Chapter 21's `[0, 1, 1, 0]`

In [8]:
import torch
import torch.nn as nn
import torch.optim as optim

torch.manual_seed(0)

class TinyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(2, 4)
        self.layer2 = nn.Linear(4, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.sigmoid(self.layer1(x))
        x = self.sigmoid(self.layer2(x))
        return x

X = torch.tensor([[0.,0.],[0.,1.],[1.,0.],[1.,1.]])
y = torch.tensor([[0.],[1.],[1.],[0.]])

proj_model = TinyNet()
proj_criterion = nn.MSELoss()
proj_optimizer = optim.Adam(proj_model.parameters(), lr=0.05)

for epoch in range(2000):
    proj_optimizer.zero_grad()
    predictions = proj_model(X)
    loss = proj_criterion(predictions, y)
    loss.backward()
    proj_optimizer.step()
    if epoch % 400 == 0:
        print(f'Epoch {epoch}, loss {loss.item():.4f}')

print(f'Final loss: {loss.item():.4f}')
print('Rounded predictions:', predictions.round().detach().view(-1).tolist())
assert loss.item() < 0.05, 'Project checklist requires final loss under 0.05'
assert predictions.round().detach().view(-1).tolist() == [0.0, 1.0, 1.0, 0.0]
print('Project checklist: PASSED (loss < 0.05, predictions match [0, 1, 1, 0])')


Epoch 0, loss 0.2544
Epoch 400, loss 0.0008


Epoch 800, loss 0.0002
Epoch 1200, loss 0.0001


Epoch 1600, loss 0.0001
Final loss: 0.0000
Rounded predictions: [0.0, 1.0, 1.0, 0.0]
Project checklist: PASSED (loss < 0.05, predictions match [0, 1, 1, 0])


## Next: Chapter 23 — Convolutional Neural Networks (Colab)

The architecture behind almost every image model — convolutions, pooling, and why CNNs need vastly fewer parameters than a fully-connected network for image data.